# GPT-2 Classification Fine-Tuning (Spam vs. Ham Detection)

This notebook demonstrates how to fine-tune a pretrained **GPT-2 (124M)** model for text classification—specifically classifying SMS messages as either **spam** or **ham** (not spam).

### Pipeline Overview
1. **Architecture Setup**: Core GPT-2 components built from scratch (`LayerNorm`, `GELU`, `MultiHeadAttention`, `TransformerBlock`, `GPTModel`).
2. **Load Pretrained Weights**: Loading base OpenAI 124M weights (reusing existing local weights without redownloading).
3. **Dataset Pipeline**: Processing the SMS Spam Collection dataset, creating train/validation/test splits, and padding via `SpamDataset`.
4. **Modify Model Architecture**: Replacing the language modeling head ($768 \rightarrow 50,257$) with a 2-class classification head ($768 \rightarrow 2$).
5. **Freeze Backbone**: Freezing base transformer blocks to preserve learned linguistic features while training only the classification head and final normalization layer.
6. **Fine-Tuning Loop**: Training with cross-entropy loss, tracking accuracy, and plotting results.
7. **Inference & Testing**: Classifying custom SMS messages in real-time.

## 1. Environment & Model Configurations

In [ ]:
import os
import math
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch version: {torch.__version__}")

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": True  # Set to True for OpenAI pretrained weights
}

## 2. Core GPT-2 Model Components (From Scratch)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=True):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / math.sqrt(self.head_dim), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"], context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

## 3. Loading Base Pretrained OpenAI Weights (Zero Redownload)

We automatically locate and load weights already downloaded in `../3. LLM Architecture/gpt2` or `../4. Fine-Tuning/gpt2`.

In [ ]:
from gpt_download import download_and_load_gpt2

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(gpt.trf_blocks[b].att.out_proj.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(gpt.trf_blocks[b].att.out_proj.bias, params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(gpt.trf_blocks[b].ff.layers[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(gpt.trf_blocks[b].ff.layers[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(gpt.trf_blocks[b].ff.layers[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(gpt.trf_blocks[b].ff.layers[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(gpt.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(gpt.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(gpt.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(gpt.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

# Load 124M base model
model = GPTModel(GPT_CONFIG_124M)
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")
load_weights_into_gpt(model, params)
model.to(device)
print("Loaded base GPT-2 124M weights into model.")

## 4. Dataset Preparation: SMS Spam Collection

We load the train, validation, and test CSV files. If not already available, we extract them from `../4. Fine-Tuning/`.

In [ ]:
# Find CSV files
csv_dirs = [".", os.path.join("..", "4. Fine-Tuning"), os.path.join("..", "3. LLM Architecture")]
train_csv, val_csv, test_csv = None, None, None
for c_dir in csv_dirs:
    if os.path.exists(os.path.join(c_dir, "train.csv")):
        train_csv = os.path.join(c_dir, "train.csv")
        val_csv = os.path.join(c_dir, "validation.csv")
        test_csv = os.path.join(c_dir, "test.csv")
        print(f"Using dataset splits from: {os.path.abspath(c_dir)}")
        break

tokenizer = tiktoken.get_encoding("gpt2")

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
        text_col = "Text" if "Text" in self.data.columns else "SMS"
        self.encoded_texts = [tokenizer.encode(text) for text in self.data[text_col]]

        if max_length is None:
            self.max_length = max(len(t) for t in self.encoded_texts)
        else:
            self.max_length = max_length
            self.encoded_texts = [t[:max_length] for t in self.encoded_texts]

        # Pad to max_length with pad_token_id (<|endoftext|>)
        self.encoded_texts = [t + [pad_token_id] * (self.max_length - len(t)) for t in self.encoded_texts]

    def __getitem__(self, idx):
        return torch.tensor(self.encoded_texts[idx], dtype=torch.long), torch.tensor(self.data.iloc[idx]["Label"], dtype=torch.long)

    def __len__(self):
        return len(self.data)

# Create data loaders
train_dataset = SpamDataset(train_csv, tokenizer, max_length=120)
val_dataset = SpamDataset(val_csv, tokenizer, max_length=train_dataset.max_length)
test_dataset = SpamDataset(test_csv, tokenizer, max_length=train_dataset.max_length)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Training set: {len(train_dataset)} examples | Batches: {len(train_loader)}")
print(f"Validation set: {len(val_dataset)} examples | Test set: {len(test_dataset)} examples")

## 5. Adding a Classification Head & Freezing the Backbone

We freeze all parameters in the Transformer blocks to keep pretrained representations intact. Then we replace `model.out_head` with a binary classification head `nn.Linear(768, 2)`.

In [ ]:
# Freeze all pretrained parameters
for param in model.parameters():
    param.requires_grad = False

# Replace output head with classification head (num_classes=2: 0=ham, 1=spam)
torch.manual_seed(123)
num_classes = 2
model.out_head = nn.Linear(GPT_CONFIG_124M["emb_dim"], num_classes)

# Make final_norm and the last transformer block trainable for better adaptation
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

model.to(device)
total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_trainable:,} out of {sum(p.numel() for p in model.parameters()):,}")

## 6. Evaluation Utilities & Training Loop

During classification, we extract the representation of the **last token** in each input sequence:

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct, total = 0, 0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)
        with torch.no_grad():
            logits = model(input_batch)[:, -1, :]  # Last token output
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == target_batch).sum().item()
        total += target_batch.numel()
    return correct / total


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # Last token output
    return torch.nn.functional.cross_entropy(logits, target_batch)


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
        else:
            break
    return total_loss / num_batches

# Evaluate initial performance before fine-tuning
initial_val_acc = calc_accuracy_loader(val_loader, model, device)
print(f"Initial Validation Accuracy (untrained head): {initial_val_acc * 100:.2f}%")

### Training the Classification Model

In [ ]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    global_step = -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
                val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1:02d} (Step {global_step:04d}): Train Loss {train_loss:.3f}, Val Loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)
        print(f"==> Epoch {epoch+1} Complete | Train Acc: {train_accuracy*100:.2f}% | Val Acc: {val_accuracy*100:.2f}%")

    return train_losses, val_losses, train_accs, val_accs

# Check if fine-tuned weights already exist to load directly
finetuned_weights = os.path.join("..", "4. Fine-Tuning", "gpt2-small124M-classification-finetuning.pth")
if os.path.exists(finetuned_weights):
    model.load_state_dict(torch.load(finetuned_weights, map_location=device))
    print(f"Loaded fine-tuned weights from: {finetuned_weights}")
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
    train_losses, val_losses, train_accs, val_accs = train_classifier_simple(
        model, train_loader, val_loader, optimizer, device, num_epochs=5, eval_freq=20, eval_iter=10
    )
    torch.save(model.state_dict(), "gpt2-small124M-classification-finetuning.pth")
    print("Saved fine-tuned model to gpt2-small124M-classification-finetuning.pth")

# Final evaluation on test set
test_acc = calc_accuracy_loader(test_loader, model, device)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

## 7. Interactive Message Classification

We test the classifier on unseen real-world SMS inputs.

In [ ]:
def classify_review(text, model, tokenizer, device, max_length=120, pad_token_id=50256):
    model.eval()
    input_ids = tokenizer.encode(text)
    input_ids = input_ids[:min(max_length, 1024)]
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
    pred = torch.argmax(probs, dim=-1).item()
    label = "spam" if pred == 1 else "not spam"
    confidence = probs[0, pred].item()
    return label, confidence

# Test on sample messages
test_messages = [
    "Hey, are we still having dinner at 7pm tonight? Let me know!",
    "WINNER!! As a valued customer you have won a $1,000 gift card. Call 08000930705 to claim now!",
    "Can you send me the lecture notes from today's class? Thanks.",
    "URGENT: Your mobile account has been credited with 500 bonus points. Text BONUS to 88088 to redeem."
]

print("--- Real-time SMS Classification Predictions ---\n")
for msg in test_messages:
    label, conf = classify_review(msg, model, tokenizer, device)
    print(f"Message: '{msg}'")
    print(f"==> Prediction: [{label.upper()}] (Confidence: {conf*100:.1f}%)\n")